# 実験音韻論の方法 ── 音韻理論をデータで検証する

### 第9章 R入門 ── 分析コード集


ここに掲載されているコードは、
『実験音韻論の方法 ── 音韻理論をデータで検証する』(音声学ライブラリ 第4巻)の
分析コード集の一部である。MIT Licenseで公開しており、自由に複製・改変・
再配布してよい(詳細はサポートリポジトリのLICENSEを参照)。

このファイルを開いた時点では閲覧用として開くが、**ファイル → ドライブに
コピーを保存**を行えば、あなた自身のGoogle Driveに保存されたあなただけの
ファイルになる。自由に変更・実行・保存してよい。

**本文で報告されている数値・図は、すべてこのノートブックのコードを実際に
実行して得られたものである。** 架空データ(シミュレーション)を使っている
箇所は、各コードセルの直前のセルにその旨を明記してある。実データを模した
説明にすぎない部分と、実際の統計的性質を確認するための実行例とを、
本書は厳格に区別しており、このノートブックもその区別をそのまま保持している。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で
言語を **R** に切り替えること。既定のPythonランタイムのままだと
以降のセルがすべてエラーになる。

**生成される図・CSVの保存場所について。** コード中に出てくる
`data/processed/`や`assets/figures/`は、あなたのGoogle Driveの中の
フォルダではない。Colabの仮想マシン上に一時的に作られるだけの作業用
フォルダであり、**セッションが切断される(一定時間の無操作、ランタイムの
再起動など)と中身ごと消える。** 保存したい図やCSVがあれば、Colab画面
左側のファイル一覧からその都度手元のパソコンにダウンロードするか、
コードを`/content/drive/...`のようなGoogle Drive上のパスに書き出すよう
自分で書き換える必要がある。


## 共通セットアップ

以降の各セルは、本書の`scripts/R/`に収録されている実際のコードと一字一句同じである(差し替えたのはこのセットアップだけ)。まずこのセルを実行してから、上から順に進めること。

In [ ]:
# ---- 図の日本語文字化け対策 ----
# Colabの既定Rランタイムのフォントには日本語グリフが無く、図中の日本語
# (タイトル・軸ラベルなど)が文字化け(豆腐記号)して表示される。showtextで
# Google Fontsの日本語フォントを取得し、"notosans"という名前で登録することで
# 回避する("sans"などの予約名はshowtextから上書きできないため専用名で登録する)。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")
showtext_auto()
showtext_opts(dpi = 100)

# ---- 共通セットアップ(scripts/R/00_setup.RのColab版) ----
# 本書のすべての解析スクリプトが冒頭で読み込む初期化処理を、Colab向けに
# 移植したもの。以降の章のコードは、本書のscripts/R/chNN_*.Rと一字一句同じ
# ものを、このセットアップの後に続けて実行する形になっている。

pkgs <- c("tidyverse", "lme4", "lmerTest", "mgcv", "itsadug",
          "phonTools", "randomForest", "patchwork", "here")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lme4)
library(lmerTest)
library(here)

set.seed(20260101)   # 乱数を使う処理(ブートストラップ、MCMC)の再現のため(第9章参照)

theme_labphon <- function(base_size = 11) {
  theme_minimal(base_size = base_size, base_family = "notosans") +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(linewidth = 0.3, colour = "grey88"),
      axis.title       = element_text(size = rel(0.95)),
      strip.text       = element_text(face = "bold", size = rel(0.9)),
      legend.position  = "bottom",
      plot.title        = element_text(face = "bold", size = rel(1.05)),
      plot.caption      = element_text(size = rel(0.8), colour = "grey35")
    )
}
theme_set(theme_labphon())

# 本書のsave_fig()はPNG/PDFをassets/figures/に書き出すが、このノートブックでは
# 画面にそのまま表示すれば十分なので、print()する版に差し替える。
# (関数のシグネチャは本書と同じなので、以降のスクリプトは書き換えなしで動く。)
save_fig <- function(plot, name, width = 5.2, height = 3.6, chapter = NULL) {
  print(plot)
  invisible(NULL)
}

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
# 一部のスクリプトはsave_fig()を経由せず直接ggsave()/png()でassets/figures/に
# 書き出す(このディレクトリ自体はColab上のこのセッション内でのみ存在する
# 一時的なものであり、書籍リポジトリの実ファイルには一切影響しない)。
for (chn in sprintf("ch%02d", 2:16)) {
  dir.create(here::here("assets", "figures", chn), recursive = TRUE, showWarnings = FALSE)
}

cat("セットアップ完了。実行環境:\n")
print(sessionInfo())


## `ch09_vowel_space.R`

対象章: 第9章「R入門」

「ggplot2の文法と音声研究の定番図」節の母音空間図。日本語5母音
(/a i u e o/)のF1・F2をシミュレートし、本文に示したコードと同じ
描画パターン(F1・F2逆軸+楕円)で母音空間図を実際に生成する。
実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260905)

# 日本語5母音のおおよその調音的位置づけに基づく、説明用のF1・F2中心値。
# 実測の音響値ではなく、母音空間図の構造(狭母音は下方、前舌母音は右方)
# を示すための仮の値である。
vowel_centers <- tibble(
  vowel = c("a", "i", "u", "e", "o"),
  F1    = c(750, 300, 350, 500, 450),
  F2    = c(1300, 2300, 1200, 1900, 900)
)

n_tokens <- 30
formants_long <- vowel_centers |>
  tidyr::crossing(token = 1:n_tokens) |>
  mutate(
    F1 = F1 + rnorm(n(), 0, 40),
    F2 = F2 + rnorm(n(), 0, 90)
  ) |>
  select(vowel, token, F1, F2)

p <- ggplot(formants_long, aes(x = F2, y = F1, color = vowel, shape = vowel)) +
  geom_point(alpha = 0.4, size = 1) +
  stat_ellipse(level = 0.68) +
  scale_x_reverse() +
  scale_y_reverse() +
  labs(x = "F2 (Hz)", y = "F1 (Hz)", color = "母音", shape = "母音",
       caption = "シミュレーションデータ。scripts/R/ch09_vowel_space.R より生成。") +
  theme_labphon()

save_fig(p, "fig09-01-vowel-space", chapter = "ch09")
cat("完了。図はassets/figures/ch09/に保存した。\n")


## `ch09_join_pitfalls.R`

対象章: 第9章「Rによるデータ処理と可視化」

「よくあるエラーとその対処」節の作例。(1) NAの伝播が要約統計量に
どう波及するか、(2) 結合キーの表記ゆれ(大文字・小文字)が left_join()
でどれだけの行を静かに欠落させるかを、実際にコードを実行して確認する。
架空のデータであり、実測データではない。
--- (1) NAの伝播 -----------------------------------------------------------

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

durations <- tibble(
  speaker  = rep(c("s01", "s02", "s03"), each = 4),
  duration = c(0.12, 0.14, 0.11, 0.13,
               0.15, NA,   0.16, 0.14,
               0.10, 0.11, 0.12, 0.09)
)

summary_no_narm <- durations |>
  group_by(speaker) |>
  summarize(mean_duration = mean(duration), .groups = "drop")

summary_narm <- durations |>
  group_by(speaker) |>
  summarize(mean_duration = mean(duration, na.rm = TRUE), .groups = "drop")

cat("na.rm指定なし:\n"); print(summary_no_narm)
cat("\nna.rm = TRUE:\n"); print(summary_narm)

# --- (2) 結合キーの表記ゆれ --------------------------------------------------
speakers_meta <- tibble(
  speaker = c("sp01", "sp02", "sp03", "sp04", "sp05"),
  gender  = c("F", "M", "F", "M", "F")
)

# 測定データ側で1名だけ大文字表記になっている(入力ミスを模す)
measures <- tibble(
  speaker  = c("sp01", "sp02", "SP03", "sp04", "sp05"),
  f0_mean  = c(210, 118, 205, 122, 198)
)

joined_naive <- measures |> left_join(speakers_meta, by = "speaker")
n_unmatched_naive <- sum(is.na(joined_naive$gender))

joined_fixed <- measures |>
  mutate(speaker = str_to_lower(speaker)) |>
  left_join(speakers_meta, by = "speaker")
n_unmatched_fixed <- sum(is.na(joined_fixed$gender))

cat("\n表記ゆれを正規化する前の未結合行数:", n_unmatched_naive, "/", nrow(measures), "\n")
cat("str_to_lower()で正規化した後の未結合行数:", n_unmatched_fixed, "/", nrow(measures), "\n")


## `ch09_seed_reproducibility.R`

対象章: 第9章「Rによるデータ処理と可視化」

「再現性のための仕組み」節の作例。乱数の種(set.seed())の有無で、
同じブートストラップ処理を2回実行した結果が一致するかどうかを、
実際に確認する。架空のデータであり、実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(tibble); library(dplyr)

toy_data <- tibble(f0 = c(210, 195, 220, 205, 198, 215, 202, 208))

bootstrap_mean <- function() {
  resampled <- sample(toy_data$f0, replace = TRUE)
  mean(resampled)
}

cat("--- シード固定なし ---\n")
run1_nofix <- bootstrap_mean()
run2_nofix <- bootstrap_mean()
cat("1回目:", run1_nofix, " 2回目:", run2_nofix,
    " 一致:", identical(run1_nofix, run2_nofix), "\n")

cat("\n--- set.seed(20260101)で固定 ---\n")
set.seed(20260101); run1_fixed <- bootstrap_mean()
set.seed(20260101); run2_fixed <- bootstrap_mean()
cat("1回目:", run1_fixed, " 2回目:", run2_fixed,
    " 一致:", identical(run1_fixed, run2_fixed), "\n")


## `ch09_csj_id_parsing.R`

対象章: 第9章「Rによるデータ処理と可視化」

「音声データに固有の整形とID設計」節の作例。CSJの講演ID(接頭辞+通し番号)
から講演種別を復元する処理を、架空のID一覧に対して実際に実行する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(tibble); library(dplyr); library(stringr)

talk_ids <- tibble(
  talk_id = c("A01M0007", "S02F1234", "A03M0102", "S04F0056")
)

talk_ids |>
  mutate(
    prefix = str_sub(talk_id, 1, 1),
    talk_type = case_when(
      prefix == "A" ~ "学会講演",
      prefix == "S" ~ "模擬講演",
      TRUE          ~ "不明"
    )
  ) |>
  print()


## `ch09_missingness_bias.R`

対象章: 第9章「Rによるデータ処理と可視化」

「欠測をどう記録するか」節の作例。欠測が言語現象と結びついている場合
(無声化した母音でF0が測定できない、など)、欠測行を静かに除外した
解析(リストワイズ除去)が、条件間の平均差をどれだけ歪めるかを、
架空のデータで確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(tibble); library(dplyr)

set.seed(20260906)
n <- 300

# 条件Bでは無声化が起きやすく(30%)、無声化するとF0が測定できない(NA)。
# 無声化したトークンは(仮に測定できたとしても)F0がやや低い母音だったとする。
condition <- rep(c("A", "B"), each = n / 2)
devoiced_prob <- ifelse(condition == "A", 0.05, 0.30)
devoiced <- rbinom(n, 1, devoiced_prob)
true_f0 <- 150 + ifelse(condition == "B", 0, 0) + rnorm(n, 0, 10) - devoiced * 8

d <- tibble(condition, devoiced, f0 = ifelse(devoiced == 1, NA, true_f0))

naive_means <- d |> group_by(condition) |> summarize(mean_f0 = mean(f0, na.rm = TRUE))
true_means  <- tibble(condition, true_f0) |> group_by(condition) |>
  summarize(mean_true_f0 = mean(true_f0))

cat("リストワイズ除去後の条件別平均F0(欠測を無視):\n")
print(naive_means)
cat("\n全トークン(欠測なし)での真の平均F0:\n")
print(true_means)
cat("\n条件間の差 --- 観測(欠測除去後):",
    round(diff(naive_means$mean_f0), 2),
    " / 真の値(欠測なし):", round(diff(true_means$mean_true_f0), 2), "\n")
